# BananaLSD: persiapan data, training, dan integrasi Flutter

Jalankan sel secara berurutan dari root proyek atau folder `training`. Persiapan data mengganti folder `dataset/`; sumber `OriginalSet/` tetap disimpan. Tahap training dan ekspor ada setelah persiapan data.


## Import dan konfigurasi

In [1]:
import hashlib
import json
import math
import os
import random
import shutil
import sys
from collections import defaultdict
from pathlib import Path

from PIL import Image

# -- konfigurasi ---------------------------------------------------------------

BASE_DIR = Path.cwd().resolve()
if not (BASE_DIR / "OriginalSet").is_dir():
    BASE_DIR = BASE_DIR / "training"
if not (BASE_DIR / "OriginalSet").is_dir():
    raise FileNotFoundError("Jalankan notebook dari root proyek atau folder training yang berisi OriginalSet.")

SEED = 42
ORIGINAL_DIR = BASE_DIR / "OriginalSet"
OUTPUT_DIR = BASE_DIR / "dataset"
SPLIT_MANIFEST = BASE_DIR / "artifacts" / "split_manifest.json"
QA_REPORT = BASE_DIR / "artifacts" / "data_qa_report.json"

CLASSES = ["cordana", "healthy", "pestalotiopsis", "sigatoka"]
VALID_EXTENSIONS = {".jpeg", ".jpg", ".png"}

TEST_RATIO = 0.20       # 20% data -> test
VAL_RATIO_OF_DEV = 0.25 # 25% dari development -> val  (sekitar 20% total)

# -- utilitas ------------------------------------------------------------------

## `pixel_hash`

In [2]:
def pixel_hash(path):
    """SHA-256 dari data piksel mentah (bukan file byte, jadi tahan metadata)."""
    try:
        img = Image.open(path)
        img.load()
        raw = img.tobytes()
        return hashlib.sha256(raw).hexdigest()
    except Exception:
        return None

## `check_integrity`

In [3]:
def check_integrity(root, classes):
    """Periksa gambar rusak, ekstensi salah, dan duplikat lintas/intra kelas."""
    corrupt = []
    bad_ext = []
    hash_to_paths = defaultdict(list)
    all_files_per_class = defaultdict(list)
    class_counts = {}

    for cls in classes:
        cls_dir = root / cls
        if not cls_dir.is_dir():
            print(f"  [PERINGATAN] Folder kelas tidak ditemukan: {cls_dir}")
            continue

        files = sorted(cls_dir.iterdir())
        class_counts[cls] = 0

        for fp in files:
            if not fp.is_file():
                continue
            rel = fp.relative_to(root).as_posix()

            if fp.suffix.lower() not in VALID_EXTENSIONS:
                bad_ext.append(rel)
                continue

            h = pixel_hash(fp)
            if h is None:
                corrupt.append(rel)
                continue

            hash_to_paths[h].append(rel)
            all_files_per_class[cls].append((rel, h))
            class_counts[cls] += 1

    duplicates = {h: paths for h, paths in hash_to_paths.items() if len(paths) > 1}

    # Deduplikasi: untuk setiap hash, simpan hanya file pertama (sorted)
    seen_hashes = set()
    valid_files = defaultdict(list)
    removed_duplicates = []

    for cls in classes:
        for rel, h in all_files_per_class.get(cls, []):
            if h in seen_hashes:
                removed_duplicates.append(rel)
                continue
            seen_hashes.add(h)
            valid_files[cls].append(rel)

    return {
        "class_counts_raw": class_counts,
        "corrupt": corrupt,
        "bad_extension": bad_ext,
        "duplicates": duplicates,
        "removed_duplicates": removed_duplicates,
        "valid_files": valid_files,
    }

## `get_file_number`

In [4]:
def get_file_number(rel_path):
    """Ambil nomor dari nama file."""
    stem = Path(rel_path).stem
    try:
        return int(stem)
    except ValueError:
        return None

## `build_leaf_groups`

In [5]:
def build_leaf_groups(file_list):
    """Kelompokkan file berdasarkan kedekatan nomor (proxy leaf-group).

    Dataset BananaLSD menamai gambar sebagai <nomor>.jpeg.
    Gambar dari daun yang sama biasanya memiliki nomor yang berdekatan
    (beberapa angle/crop dari satu daun). Kita kelompokkan file yang
    nomornya berselisih <= 1 menjadi satu grup.

    Agar tidak terbentuk satu grup raksasa, batasi ukuran
    maksimum grup menjadi 5 gambar.
    """
    MAX_GROUP_SIZE = 5
    MAX_GAP = 1

    numbered = []
    non_numbered = []

    for f in file_list:
        num = get_file_number(f)
        if num is not None:
            numbered.append((num, f))
        else:
            non_numbered.append(f)

    numbered.sort(key=lambda x: x[0])

    groups = []
    current_group = []
    prev_num = None

    for num, f in numbered:
        if prev_num is not None and (num - prev_num > MAX_GAP or len(current_group) >= MAX_GROUP_SIZE):
            groups.append(current_group)
            current_group = []
        current_group.append(f)
        prev_num = num

    if current_group:
        groups.append(current_group)

    for f in non_numbered:
        groups.append([f])

    return groups

## `split_groups_by_ratio`

In [6]:
def split_groups_by_ratio(groups, ratio, rng):
    """Split grup secara acak, mengembalikan (bagian_a, bagian_b).
    bagian_b mendapat sekitar `ratio` dari total file."""
    rng.shuffle(groups)
    total = sum(len(g) for g in groups)
    target_b = math.ceil(total * ratio)

    part_b = []
    part_a = []
    count_b = 0

    for g in groups:
        if count_b < target_b:
            part_b.extend(g)
            count_b += len(g)
        else:
            part_a.extend(g)

    return part_a, part_b

## `copy_split`

In [7]:
def copy_split(root, output_dir, split_name, cls, files):
    """Salin file ke folder output/<split>/<kelas>/."""
    dest = output_dir / split_name / cls
    dest.mkdir(parents=True, exist_ok=True)
    for rel in files:
        src = root / rel
        shutil.copy2(src, dest / src.name)

## Alur persiapan data

In [8]:
def main():
    print("=" * 64)
    print("  PERSIAPAN DATASET BananaLSD")
    print("=" * 64)

    if not ORIGINAL_DIR.is_dir():
        print(f"[ERROR] Folder OriginalSet tidak ditemukan: {ORIGINAL_DIR}")
        sys.exit(1)

    # -- langkah 1-3: pemeriksaan kualitas data ----------------------------
    print("\n[1/6] Memeriksa integritas gambar ...")
    qa = check_integrity(ORIGINAL_DIR, CLASSES)

    print(f"\n  Jumlah gambar per kelas (sebelum deduplikasi):")
    for cls in CLASSES:
        n = qa["class_counts_raw"].get(cls, 0)
        print(f"    {cls:20s}: {n}")
    print(f"    {'TOTAL':20s}: {sum(qa['class_counts_raw'].values())}")

    if qa["corrupt"]:
        print(f"\n  [!] Gambar rusak ({len(qa['corrupt'])}):")
        for f in qa["corrupt"]:
            print(f"    - {f}")
    else:
        print("\n  [OK] Tidak ada gambar rusak.")

    if qa["bad_extension"]:
        print(f"\n  [!] Ekstensi tidak valid ({len(qa['bad_extension'])}):")
        for f in qa["bad_extension"]:
            print(f"    - {f}")
    else:
        print("\n  [OK] Semua ekstensi valid.")

    n_dup_groups = len(qa["duplicates"])
    n_dup_files = sum(len(v) for v in qa["duplicates"].values())
    if qa["duplicates"]:
        print(f"\n  [!] Duplikat terdeteksi: {n_dup_groups} grup, {n_dup_files} file total")
        for h, paths in list(qa["duplicates"].items())[:5]:
            print(f"    hash ..{h[:12]}:")
            for p in paths:
                print(f"      - {p}")
        if n_dup_groups > 5:
            print(f"    ... dan {n_dup_groups - 5} grup lainnya.")
    else:
        print("\n  [OK] Tidak ada duplikat.")

    # -- langkah 2: deduplikasi --------------------------------------------
    print(f"\n[2/6] Deduplikasi ...")
    print(f"  Duplikat dihapus: {len(qa['removed_duplicates'])} file")

    total_valid = 0
    print(f"\n  Jumlah gambar per kelas (setelah deduplikasi):")
    for cls in CLASSES:
        n = len(qa["valid_files"].get(cls, []))
        total_valid += n
        print(f"    {cls:20s}: {n}")
    print(f"    {'TOTAL':20s}: {total_valid}")

    # -- langkah 3-4: pembagian data ---------------------------------------
    print("\n[3/6] Membangun leaf-group dan membagi data ...")
    rng = random.Random(SEED)

    manifest = {
        "seed": SEED,
        "ratios": {
            "test": TEST_RATIO,
            "val_of_dev": VAL_RATIO_OF_DEV,
        },
        "splits": {},
    }

    summary_rows = []

    for cls in CLASSES:
        files = qa["valid_files"].get(cls, [])
        if not files:
            continue

        # Langkah 1: bagi menjadi dev (80%) dan test (20%)
        groups = build_leaf_groups(files)
        dev_files, test_files = split_groups_by_ratio(groups, TEST_RATIO, rng)

        # Langkah 2: bagi dev menjadi train (75%) dan val (25%)
        dev_groups = build_leaf_groups(dev_files)
        train_files, val_files = split_groups_by_ratio(dev_groups, VAL_RATIO_OF_DEV, rng)

        manifest["splits"][cls] = {
            "train": sorted(train_files),
            "val": sorted(val_files),
            "test": sorted(test_files),
        }

        summary_rows.append({
            "class": cls,
            "total": len(files),
            "train": len(train_files),
            "val": len(val_files),
            "test": len(test_files),
        })

    # tabel ringkasan
    print(f"\n  {'Kelas':20s} {'Total':>6s} {'Train':>6s} {'Val':>6s} {'Test':>6s}")
    print("  " + "-" * 44)
    t_total = t_train = t_val = t_test = 0
    for r in summary_rows:
        print(
            f"  {r['class']:20s} {r['total']:6d} {r['train']:6d} "
            f"{r['val']:6d} {r['test']:6d}"
        )
        t_total += r["total"]
        t_train += r["train"]
        t_val += r["val"]
        t_test += r["test"]
    print("  " + "-" * 44)
    print(f"  {'TOTAL':20s} {t_total:6d} {t_train:6d} {t_val:6d} {t_test:6d}")
    if t_total > 0:
        print(
            f"  {'RASIO':20s} {'':>6s} "
            f"{t_train/t_total:.1%}  {t_val/t_total:.1%}  {t_test/t_total:.1%}"
        )

    # -- langkah 5: verifikasi tidak ada kebocoran antar split -------------
    print("\n[4/6] Memverifikasi tidak ada kebocoran antar split ...")
    leak_found = False
    for cls in CLASSES:
        sp = manifest["splits"].get(cls, {})
        sets = {name: set(sp.get(name, [])) for name in ["train", "val", "test"]}
        for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
            overlap = sets[a] & sets[b]
            if overlap:
                print(f"  [!] KEBOCORAN {cls}: {a} & {b} = {len(overlap)} file")
                leak_found = True
    if not leak_found:
        print("  [OK] Tidak ada kebocoran -- setiap gambar hanya ada di satu split.")

    # -- langkah 6: simpan manifest ----------------------------------------
    print("\n[5/6] Menyimpan manifest dan laporan QA ...")
    SPLIT_MANIFEST.parent.mkdir(parents=True, exist_ok=True)

    with open(SPLIT_MANIFEST, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2, ensure_ascii=False)
    print(f"  [OK] Manifest disimpan: {SPLIT_MANIFEST}")

    qa_report = {
        "class_counts_raw": qa["class_counts_raw"],
        "total_valid_after_dedup": total_valid,
        "corrupt_files": qa["corrupt"],
        "bad_extension_files": qa["bad_extension"],
        "duplicate_groups": {h: paths for h, paths in qa["duplicates"].items()},
        "removed_duplicates": qa["removed_duplicates"],
        "split_summary": summary_rows,
    }
    with open(QA_REPORT, "w", encoding="utf-8") as f:
        json.dump(qa_report, f, indent=2, ensure_ascii=False)
    print(f"  [OK] Laporan QA disimpan: {QA_REPORT}")

    # -- langkah 7: salin file ke folder split -----------------------------
    print("\n[6/6] Menyalin file ke folder dataset/ ...")
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)

    for cls in CLASSES:
        sp = manifest["splits"].get(cls, {})
        for split_name in ["train", "val", "test"]:
            files = sp.get(split_name, [])
            copy_split(ORIGINAL_DIR, OUTPUT_DIR, split_name, cls, files)
            print(f"  [OK] {split_name}/{cls}: {len(files)} file disalin")
    print("\n" + "=" * 64)
    print("  PERSIAPAN DATA SELESAI")
    print("=" * 64)
    print(f"\n  Dataset siap di: {OUTPUT_DIR}")
    print(f"  Manifest split:  {SPLIT_MANIFEST}")
    print(f"  Laporan QA:      {QA_REPORT}")

## Jalankan persiapan data

Sel ini menulis laporan ke `artifacts/` dan mengganti isi folder `dataset/` dengan hasil pembagian terbaru.

In [9]:
main()

  PERSIAPAN DATASET BananaLSD

[1/6] Memeriksa integritas gambar ...


c:\PBL-KEL4_TI3C_SEM5\.venv\Lib\site-packages\PIL\TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
c:\PBL-KEL4_TI3C_SEM5\.venv\Lib\site-packages\PIL\JpegImagePlugin.py:874: UserWarning: Image appears to be a malformed MPO file, it will be interpreted as a base JPEG file
  warnings.warn(



  Jumlah gambar per kelas (sebelum deduplikasi):
    cordana             : 162
    healthy             : 129
    pestalotiopsis      : 173
    sigatoka            : 473
    TOTAL               : 937

  [OK] Tidak ada gambar rusak.

  [OK] Semua ekstensi valid.

  [!] Duplikat terdeteksi: 39 grup, 78 file total
    hash ..ece18024954d:
      - healthy/0.jpeg
      - healthy/104.jpeg
    hash ..e9d9a7a09fa6:
      - healthy/1.jpeg
      - healthy/109.jpeg
    hash ..378b21d4fb3c:
      - healthy/10.jpeg
      - healthy/106.jpeg
    hash ..2c8f97da3c3f:
      - healthy/101.jpeg
      - healthy/46.jpeg
    hash ..3b1c6d5d004a:
      - healthy/102.jpeg
      - healthy/39.jpeg
    ... dan 34 grup lainnya.

[2/6] Deduplikasi ...
  Duplikat dihapus: 39 file

  Jumlah gambar per kelas (setelah deduplikasi):
    cordana             : 162
    healthy             : 91
    pestalotiopsis      : 172
    sigatoka            : 473
    TOTAL               : 898

[3/6] Membangun leaf-group dan membagi 

## 1. Training sampai Flutter: urutan menjalankan

Jalankan notebook dari atas ke bawah. Bagian sebelumnya memeriksa dan membagi dataset.
Jika split sudah dibuat dan tidak ingin menggantinya, lewati hanya sel `main()`.
Tahapan berikut: muat train/val/test → training → evaluasi test → simpan Keras →
konversi dan verifikasi TFLite → salin aset → jalankan Flutter Android.

Kernel membutuhkan `tensorflow`, `numpy`, dan `Pillow`. Jika belum tersedia, jalankan
`%pip install tensorflow numpy Pillow` di sel terpisah lalu restart kernel.
MobileNetV2 mengunduh bobot ImageNet pada pemakaian pertama. Training CPU dapat memakan waktu.
Test tidak dipakai untuk memilih epoch. Pengelompokan berdasarkan nomor file di atas
belum menjamin pemisahan identitas daun; konfirmasi identitas daun sebelum mengklaim generalisasi.

In [10]:
import tensorflow as tf
import numpy as np
from PIL import ImageOps

tf.keras.utils.set_random_seed(SEED)
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 20
ARTIFACT_DIR = BASE_DIR / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_PATH = ARTIFACT_DIR / "banana_best.keras"
FLUTTER_ASSET_DIR = BASE_DIR.parent / "ecoplant" / "assets" / "models"
print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


## 2. Muat dataset dan tetapkan urutan label

Label berasal dari urutan `CLASSES`, bukan urutan enum Flutter. Input RGB float32
berukuran 224 × 224 dalam rentang 0–255. Normalisasi ke [-1, 1] ada di dalam model;
Flutter tidak boleh membagi piksel dengan 255 lagi. Resize nearest memakai koordinat
`floor(indeks * ukuran_asli / 224)` di Python dan Dart agar konsisten.

In [11]:
class_names = list(CLASSES)
labels = [name.capitalize() for name in class_names]
class_indices = {name: index for index, name in enumerate(class_names)}
print(class_indices)

def load_rgb(path):
    with Image.open(path) as source:
        pixels = np.asarray(ImageOps.exif_transpose(source).convert("RGB"))
    ys = np.arange(IMG_SIZE) * pixels.shape[0] // IMG_SIZE
    xs = np.arange(IMG_SIZE) * pixels.shape[1] // IMG_SIZE
    return pixels[ys[:, None], xs[None, :]].astype(np.float32)

split_paths = {}
split_targets = {}
for split in ("train", "val", "test"):
    paths, targets = [], []
    for name, index in class_indices.items():
        files = sorted(p for p in (OUTPUT_DIR / split / name).glob("*")
                       if p.suffix.lower() in VALID_EXTENSIONS)
        if not files:
            raise ValueError(f"Dataset kosong: {split}/{name}. Jalankan persiapan data.")
        paths.extend(files)
        targets.extend([index] * len(files))
    split_paths[split], split_targets[split] = paths, targets
    print(split, len(paths), "gambar")

def make_dataset(split):
    def samples():
        for path, target in zip(split_paths[split], split_targets[split]):
            yield load_rgb(path), np.int32(target)
    ds = tf.data.Dataset.from_generator(samples, output_signature=(
        tf.TensorSpec((IMG_SIZE, IMG_SIZE, 3), tf.float32),
        tf.TensorSpec((), tf.int32)))
    ds = ds.apply(tf.data.experimental.assert_cardinality(len(split_paths[split])))
    if split == "train":
        ds = ds.shuffle(len(split_paths[split]), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds, test_ds = [make_dataset(s) for s in ("train", "val", "test")]

{'cordana': 0, 'healthy': 1, 'pestalotiopsis': 2, 'sigatoka': 3}
train 528 gambar
val 181 gambar
test 189 gambar


## 3. Buat model dan training

Backbone ImageNet dibekukan. Augmentasi hanya aktif saat training. Checkpoint memilih validation loss terbaik.

In [12]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
], name="augmentation")
backbone = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
backbone.trainable = False
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="rgb")
x = augmentation(inputs)
x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1)(x)
x = backbone(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(len(labels), activation="softmax", name="probabilities")(x)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, shuffle=False, callbacks=[
    tf.keras.callbacks.ModelCheckpoint(str(BEST_MODEL_PATH), save_best_only=True,
                                       monitor="val_loss"),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                     restore_best_weights=True),
    tf.keras.callbacks.CSVLogger(str(ARTIFACT_DIR / "training_history.csv")),
])

Epoch 1/20


c:\PBL-KEL4_TI3C_SEM5\.venv\Lib\site-packages\PIL\TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
c:\PBL-KEL4_TI3C_SEM5\.venv\Lib\site-packages\PIL\JpegImagePlugin.py:874: UserWarning: Image appears to be a malformed MPO file, it will be interpreted as a base JPEG file
  warnings.warn(


33/33 ━━━━━━━━━━━━━━━━━━━━ 22s 321ms/step - accuracy: 0.5644 - loss: 1.1772 - val_accuracy: 0.8011 - val_loss: 0.5383
Epoch 2/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 9s 246ms/step - accuracy: 0.8712 - loss: 0.3804 - val_accuracy: 0.9116 - val_loss: 0.3385
Epoch 3/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 10s 268ms/step - accuracy: 0.9167 - loss: 0.2552 - val_accuracy: 0.9116 - val_loss: 0.2873
Epoch 4/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 9s 249ms/step - accuracy: 0.9545 - loss: 0.1918 - val_accuracy: 0.9337 - val_loss: 0.2608
Epoch 5/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 9s 238ms/step - accuracy: 0.9527 - loss: 0.1805 - val_accuracy: 0.9337 - val_loss: 0.2400
Epoch 6/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 219ms/step - accuracy: 0.9527 - loss: 0.1447 - val_accuracy: 0.9448 - val_loss: 0.2260
Epoch 7/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 219ms/step - accuracy: 0.9697 - loss: 0.1195 - val_accuracy: 0.9337 - val_loss: 0.2070
Epoch 8/20
33/33 ━━━━━━━━━━━━━━━━━━━━ 8s 211ms/step - accuracy: 0.9697 - loss: 0.1173 - val_accuracy: 0.9227 - va

## 4. Evaluasi model terbaik dan simpan

Confusion matrix: baris = kelas sebenarnya, kolom = prediksi, sesuai urutan labels. Nilai evaluasi bukan jaminan performa foto lapangan.

In [13]:
model = tf.keras.models.load_model(BEST_MODEL_PATH)
test_metrics = model.evaluate(test_ds, return_dict=True)
probabilities = model.predict(test_ds)
predictions = probabilities.argmax(axis=1)
confusion = tf.math.confusion_matrix(split_targets["test"], predictions,
                                    num_classes=len(labels)).numpy()
print("Urutan label:", labels)
print("Confusion matrix:\n", confusion)
report = {"metrics": {k: float(v) for k, v in test_metrics.items()},
          "labels": labels, "confusion_matrix": confusion.tolist()}
(ARTIFACT_DIR / "evaluation.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
model.save(ARTIFACT_DIR / "banana_model.keras")

12/12 ━━━━━━━━━━━━━━━━━━━━ 5s 278ms/step - accuracy: 0.9365 - loss: 0.1749
12/12 ━━━━━━━━━━━━━━━━━━━━ 3s 204ms/step
Urutan label: ['Cordana', 'Healthy', 'Pestalotiopsis', 'Sigatoka']
Confusion matrix:
 [[33  0  1  1]
 [ 0 18  2  1]
 [ 0  1 33  1]
 [ 0  1  4 93]]


## 5. Konversi dan verifikasi TFLite

Ekspor float32 dengan operasi TFLite bawaan. Periksa bentuk tensor dan bandingkan prediksi Keras/TFLite sebelum menyalin aset. Tidak perlu training ulang untuk ekspor: jalankan konfigurasi, dataset, lalu muat checkpoint pada tahap 4.

In [14]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
tflite_model = converter.convert()
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()
input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]
assert input_info["shape"].tolist() == [1, IMG_SIZE, IMG_SIZE, 3]
assert output_info["shape"].tolist() == [1, len(labels)]
assert input_info["dtype"] == output_info["dtype"] == np.float32
max_error = 0.0
for path in split_paths["test"][:10]:
    sample = load_rgb(path)[None, ...]
    expected = model(sample, training=False).numpy()
    interpreter.set_tensor(input_info["index"], sample)
    interpreter.invoke()
    actual = interpreter.get_tensor(output_info["index"])
    np.testing.assert_allclose(actual, expected, atol=1e-4, rtol=1e-3)
    max_error = max(max_error, float(np.max(np.abs(actual - expected))))
print("Selisih maksimum Keras/TFLite:", max_error)
(ARTIFACT_DIR / "model.tflite").write_bytes(tflite_model)
(ARTIFACT_DIR / "labels.json").write_text(json.dumps(labels, indent=2), encoding="utf-8")
metadata = {"version": "banana-" + hashlib.sha256(tflite_model).hexdigest()[:12],
            "input_shape": [1, IMG_SIZE, IMG_SIZE, 3], "input_dtype": "float32",
            "pixel_range": [0, 255], "resize": "nearest_floor", "color": "RGB",
            "normalization": "inside_model", "labels": labels,
            "tensorflow_version": tf.__version__, "test_metrics": report["metrics"]}
(ARTIFACT_DIR / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

INFO:tensorflow:Assets written to: C:\Users\rizal\AppData\Local\Temp\tmpov09i_40\assets


INFO:tensorflow:Assets written to: C:\Users\rizal\AppData\Local\Temp\tmpov09i_40\assets


Saved artifact at 'C:\Users\rizal\AppData\Local\Temp\tmpov09i_40'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='rgb')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  2347298334928: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2347298335696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2347298336272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2347298335888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2347298333968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2347298336848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2347298337040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2347298334736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2347298336656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2347298337616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2347298332816:

c:\PBL-KEL4_TI3C_SEM5\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Selisih maksimum Keras/TFLite: 2.9206275939941406e-06


455

## 6. Salin model ke Flutter

Sel ini mengganti tiga aset model Flutter dengan hasil ekspor terbaru. Simpan model Keras di artifacts untuk training lanjutan; Flutter hanya memakai TFLite, label, dan metadata.

In [15]:
if not (BASE_DIR.parent / "ecoplant" / "pubspec.yaml").is_file():
    raise FileNotFoundError("Folder proyek Flutter ecoplant tidak ditemukan.")
FLUTTER_ASSET_DIR.mkdir(parents=True, exist_ok=True)
for filename in ("model.tflite", "labels.json", "model_metadata.json"):
    shutil.copy2(ARTIFACT_DIR / filename, FLUTTER_ASSET_DIR / filename)
    print("Disalin:", FLUTTER_ASSET_DIR / filename)

Disalin: C:\PBL-KEL4_TI3C_SEM5\ecoplant\assets\models\model.tflite
Disalin: C:\PBL-KEL4_TI3C_SEM5\ecoplant\assets\models\labels.json
Disalin: C:\PBL-KEL4_TI3C_SEM5\ecoplant\assets\models\model_metadata.json


## 7. Jalankan dan uji di Flutter Android

Integrasi sudah ada di `ecoplant/lib/features/detection/tflite_detection_service.dart`.
Provider analisis memuat aset, membaca RGB, memperbaiki orientasi EXIF, resize 224 × 224,
menjalankan model di isolate, lalu memetakan argmax ke label JSON dan confidence.

Setelah sel penyalinan berhasil, jalankan dari terminal root proyek:

```powershell
cd ecoplant
flutter pub get
flutter devices
flutter run -d <id-perangkat-android>
```

Gunakan Android API 26 atau lebih baru. Buka scan, pilih foto daun dari galeri atau kamera,
lalu analisis. Pastikan hasil menampilkan kelas dan confidence. Bandingkan beberapa foto
test dengan prediksi Python. Saat mengganti model lakukan stop dan run ulang, bukan hanya hot reload.
Jika aset belum ada, aplikasi menampilkan pesan kegagalan pemuatan model.
Target awal integrasi ini Android; Flutter Web tidak didukung oleh paket TFLite ini,
sedangkan desktop memerlukan library native tambahan. Pengujian perangkat tetap diperlukan.

Referensi: [MobileNetV2](https://www.tensorflow.org/api_docs/python/tf/keras/applications/MobileNetV2),
[konversi TFLite](https://www.tensorflow.org/api_docs/python/tf/lite/TFLiteConverter),
[tflite_flutter](https://pub.dev/packages/tflite_flutter).